# Fine-tuning GPT-2 cho Task Adaptation

## Mô hình trước vs. Sau khi fine-tuning trên NLI

---

```
GPT-2 pretrained
     ↓
Fine-tune trên NLI task
     ↓
So sánh: Before FT vs After FT
```

**Mục tiêu:** Trình diễn sự cải thiện của mô hình GPT-2 sau khi fine-tuning trên tác vụ Natural Language Inference (NLI) — cùng kiến trúc, cùng prompt formatting, chỉ khác trọng số mô hình.

> Notebook này là bản mở rộng của `gpt_task_adaptation_demo.ipynb`. Bản gốc trình diễn zero-shot task transformation; bản này bổ sung fine-tuning và so sánh trước/sau.


In [ ]:
# @title Cài đặt & Thiết lập

import sys, os, warnings, logging, math
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Tự động cài đặt nếu đang chạy trên Google Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install -q transformers datasets torch matplotlib
except ImportError:
    IN_COLAB = False

print('=' * 60)
print('Thiết lập thư viện...')

import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import load_dataset
import matplotlib.pyplot as plt

# ---- Cấu hình (config) ----
print('Cấu hình...')

# Subset size (điều chỉnh nếu chạy trên CPU hoặc muốn nhanh hơn)
USE_LARGER_SUBSET = False

if USE_LARGER_SUBSET and torch.cuda.is_available():
    NLI_TRAIN_SIZE = 3000
    NLI_VAL_SIZE = 500
    NUM_EPOCHS = 3
else:
    NLI_TRAIN_SIZE = 1000
    NLI_VAL_SIZE = 200
    NUM_EPOCHS = 2

# Similarity (optional, mặc định tắt)
RUN_SIMILARITY = False
SIM_TRAIN_SIZE = 1000
SIM_VAL_SIZE = 200
SIM_THRESHOLD = 3.5

# Huấn luyện
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 5e-5
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

print(f'  NLI train:     {NLI_TRAIN_SIZE}')
print(f'  NLI val:       {NLI_VAL_SIZE}')
print(f'  Epochs:        {NUM_EPOCHS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Grad accum:    {GRAD_ACCUM_STEPS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Max length:    {MAX_LENGTH}')
print()

# ---- Load model & tokenizer ----
print('Đang tải GPT-2 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('gpt2')

print('Đang tải GPT-2 model...')
model = AutoModelForCausalLM.from_pretrained('gpt2')

# ---- Special tokens ----
print()
print('Đang đăng ký special tokens: <s>, $, <e>')
tokenizer.add_special_tokens({
    'additional_special_tokens': ['<s>', '$', '<e>']
})
model.resize_token_embeddings(len(tokenizer))

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Tổng số tokens: {len(tokenizer)}')
print(f'Pad token:      {tokenizer.pad_token} (id={tokenizer.pad_token_id})')
print('Ready ✓')
print('=' * 60)

# ---- Label maps ----
NLI_LABEL_MAP = {0: 'Entailment', 1: 'Neutral', 2: 'Contradiction'}
NLI_LABEL_IDS = {v: k for k, v in NLI_LABEL_MAP.items()}

if RUN_SIMILARITY:
    SIM_LABEL_MAP = {0: 'Not Similar', 1: 'Similar'}
    # STS-B label is a float 0-5, we convert

print(f'NLI labels: {NLI_LABEL_MAP}')
print('=' * 60)


---

## Dataset: MultiNLI

- **Nguồn:** [nyu-mll/multi_nli](https://huggingface.co/datasets/nyu-mll/multi_nli)
- **Tác vụ:** Natural Language Inference (NLI)
- **Dạng:** cặp (premise, hypothesis) + label

### Ví dụ

```
Premise:    A man is playing guitar.
Hypothesis: A person is making music.
Label:      Entailment
```

↓

### Prompt huấn luyện

```
<s>
Premise: A man is playing guitar.
$
Hypothesis: A person is making music.

Choices:
Entailment
Neutral
Contradiction

Relationship:
 Entailment
<e>
```

> Chiến lược: mask loss trên toàn bộ prompt (các dòng trước `Relationship:`) và chỉ supervise label tokens (`Entailment\n<e>`).

---


In [ ]:
# @title Trước fine-tuning — dự đoán trên NLI

def format_nli_prompt(premise, hypothesis):
    return (
        f'<s>\n'
        f'Premise: {premise}\n'
        f'$\n'
        f'Hypothesis: {hypothesis}\n'
        f'\n'
        f'Choices:\n'
        f'Entailment\n'
        f'Neutral\n'
        f'Contradiction\n'
        f'\n'
        f'Relationship:\n'
    )


def score_nli_choices(model, tokenizer, prompt_text):
    """Tính log-probability đầy đủ cho mỗi lựa chọn."""
    device = model.device
    inputs = tokenizer(prompt_text, return_tensors='pt').to(device)
    input_len = inputs.input_ids.shape[1]

    choices = ['Entailment', 'Neutral', 'Contradiction']
    scores = {}

    for choice in choices:
        # Label text: space + label + newline + <e>
        label_text = f' {choice}\n<e>'
        label_ids = tokenizer.encode(label_text, add_special_tokens=False)
        label_tensor = torch.tensor([label_ids]).to(device)

        full_ids = torch.cat([inputs.input_ids, label_tensor], dim=1)

        with torch.no_grad():
            outputs = model(full_ids)
            logits = outputs.logits[0]  # (seq_len, vocab)

        # Tổng log-probability của label sequence
        total_log_prob = 0.0
        for k, lid in enumerate(label_ids):
            pos = input_len + k - 1  # logits[pos] predicts token at pos+1
            log_probs = torch.log_softmax(logits[pos], dim=-1)
            total_log_prob += log_probs[lid].item()

        # Chuẩn hoá theo độ dài label
        scores[choice] = total_log_prob / len(label_ids)

    prediction = max(scores, key=scores.get)
    return prediction, scores


def evaluate_accuracy(model, tokenizer, dataset, num_samples=100):
    """Đánh giá accuracy trên subset."""
    correct = 0
    total = min(num_samples, len(dataset))
    for i in range(total):
        ex = dataset[i]
        prompt = format_nli_prompt(ex['premise'], ex['hypothesis'])
        pred, _ = score_nli_choices(model, tokenizer, prompt)
        true_label = NLI_LABEL_MAP[ex['label']]
        if pred == true_label:
            correct += 1
    return correct / total


# === Demo examples ===
print('=' * 60)
print('TRUOC FINE-TUNING: Dự đoán trên NLI')
print('=' * 60)
print()

demo_examples = [
    ('A man is playing guitar.', 'A person is making music.', 'Entailment'),
    ('The cat sat on the mat.', 'The dog ran in the park.', 'Neutral'),
    ('It is raining heavily outside.', 'The ground is dry.', 'Contradiction'),
]

for premise, hypothesis, true_label in demo_examples:
    print(f'Premise:    {premise}')
    print(f'Hypothesis: {hypothesis}')
    print(f'True label: {true_label}')

    prompt = format_nli_prompt(premise, hypothesis)
    pred, scores = score_nli_choices(model, tokenizer, prompt)

    print('Scores:')
    for choice, score in sorted(scores.items(), key=lambda x: -x[1]):
        marker = '  ←' if choice == pred else ''
        print(f'  {choice:15s}: {score:.4f}{marker}')
    print(f'Prediction: {pred}')
    print(f'Correct:    {pred == true_label}')
    print()

# === Đánh giá accuracy trên 100 mẫu validation ===
print('-' * 40)
print('Đang tải validation set để đánh giá accuracy...')
val_preview = load_dataset('nyu-mll/multi_nli', split=f'validation_matched[:{NLI_VAL_SIZE}]')
acc_before = evaluate_accuracy(model, tokenizer, val_preview, num_samples=100)
print(f'Accuracy BEFORE fine-tuning: {acc_before:.2%} (random baseline: ~33%)')
print('=' * 60)


In [ ]:
# @title Tiền xử lý dữ liệu huấn luyện

print('=' * 60)
print('Tải và tiền xử lý dữ liệu...')

# --- Load dataset ---
train_raw = load_dataset('nyu-mll/multi_nli', split=f'train[:{NLI_TRAIN_SIZE}]')
val_raw = load_dataset('nyu-mll/multi_nli', split=f'validation_matched[:{NLI_VAL_SIZE}]')

print(f'Train: {len(train_raw)} samples')
print(f'Validation: {len(val_raw)} samples')
print()

# --- Format & tokenize ---
def format_nli_label(label_id):
    return f' {NLI_LABEL_MAP[label_id]}\n<e>'


def tokenize_nli(examples):
    """
    Tokenize và tạo labels với masked loss.
    - prompt tokens → labels = -100 (bỏ qua loss)
    - label tokens → labels = token id (học để sinh label)
    """
    prompts = [
        format_nli_prompt(p, h)
        for p, h in zip(examples['premise'], examples['hypothesis'])
    ]
    label_texts = [format_nli_label(l) for l in examples['label']]

    all_input_ids, all_labels, all_attn = [], [], []

    for prompt, label_text in zip(prompts, label_texts):
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
        label_ids = tokenizer.encode(label_text, add_special_tokens=False)

        # Giới hạn prompt để chứa được label
        max_prompt_len = MAX_LENGTH - len(label_ids)
        if len(prompt_ids) > max_prompt_len:
            prompt_ids = prompt_ids[:max_prompt_len]

        input_ids = prompt_ids + label_ids
        attn_mask = [1] * len(input_ids)
        labels = ([-100] * len(prompt_ids)) + label_ids

        all_input_ids.append(input_ids)
        all_labels.append(labels)
        all_attn.append(attn_mask)

    return {
        'input_ids': all_input_ids,
        'labels': all_labels,
        'attention_mask': all_attn,
    }


print('Tokenizing train set...')
train_data = train_raw.map(
    tokenize_nli,
    batched=True,
    remove_columns=train_raw.column_names,
)
print('Tokenizing validation set...')
val_data = val_raw.map(
    tokenize_nli,
    batched=True,
    remove_columns=val_raw.column_names,
)

print()
print(f'Train tokens:  {len(train_data)} examples')
print(f'Val tokens:    {len(val_data)} examples')

# Kiểm tra mẫu
sample = train_data[0]
print(f'\nSample input_ids length: {len(sample["input_ids"])}')
print(f'Sample labels length:     {len(sample["labels"])}')
n_masked = sum(1 for l in sample['labels'] if l == -100)
print(f'Masked prompt tokens:     {n_masked}')
print(f'Supervised label tokens:  {len(sample["labels"]) - n_masked}')
print()

# --- Data collator ---
def collate_fn(batch):
    """Pad batch tới độ dài lớn nhất. Labels pad bằng -100."""
    input_ids = [item['input_ids'] for item in batch]
    labels = [item['labels'] for item in batch]
    attn_mask = [item['attention_mask'] for item in batch]

    max_len = max(len(x) for x in input_ids)

    def pad(seqs, val):
        return [s + [val] * (max_len - len(s)) for s in seqs]

    return {
        'input_ids': torch.tensor(pad(input_ids, tokenizer.pad_token_id)),
        'labels': torch.tensor(pad(labels, -100)),
        'attention_mask': torch.tensor(pad(attn_mask, 0)),
    }

print('Data collator ready.')
print('=' * 60)


In [ ]:
# @title Cấu hình Trainer

print('=' * 60)
print('Training Hyperparameters')
print('=' * 60)

training_args = TrainingArguments(
    output_dir='./training_output',
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    logging_steps=50,
    logging_first_step=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    dataloader_drop_last=True,
    remove_unused_columns=False,
    report_to='none',
)

print(f'  Batch size:            {BATCH_SIZE}')
print(f'  Gradient accum steps:  {GRAD_ACCUM_STEPS}')
print(f'  Effective batch size:  {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Learning rate:         {LEARNING_RATE}')
print(f'  Weight decay:          {WEIGHT_DECAY}')
print(f'  Warmup ratio:          {WARMUP_RATIO}')
print(f'  Epochs:                {NUM_EPOCHS}')
print(f'  Max length:            {MAX_LENGTH}')
print(f'  FP16:                  {torch.cuda.is_available()}')
print()

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')

print('=' * 60)


In [ ]:
# @title Huấn luyện

print('=' * 60)
print('BẮT ĐẦU FINE-TUNING')
print('=' * 60)
print()
print(f'Train samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()

# --- Vẽ loss curve ---
logs = trainer.state.log_history

# Tách train steps và eval steps
train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for log in logs:
    if 'loss' in log and 'eval_loss' not in log:
        train_steps.append(log.get('step', len(train_steps)))
        train_losses.append(log['loss'])
    if 'eval_loss' in log:
        eval_steps.append(log.get('step', len(eval_steps)))
        eval_losses.append(log['eval_loss'])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train loss', alpha=0.7)
if eval_steps:
    plt.plot(eval_steps, eval_losses, label='Eval loss', marker='o', linewidth=2)
plt.xlabel('Training steps')
plt.ylabel('Loss')
plt.title('Loss during fine-tuning')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print()
final_eval = trainer.evaluate()
print(f'Final eval loss: {final_eval["eval_loss"]:.4f}')
print('=' * 60)


In [ ]:
# @title Sau fine-tuning — dự đoán & so sánh

print('=' * 60)
print('SAU FINE-TUNING: Dự đoán trên NLI')
print('=' * 60)
print()

for premise, hypothesis, true_label in demo_examples:
    print(f'Premise:    {premise}')
    print(f'Hypothesis: {hypothesis}')
    print(f'True label: {true_label}')

    prompt = format_nli_prompt(premise, hypothesis)
    pred, scores = score_nli_choices(model, tokenizer, prompt)

    print('Scores:')
    for choice, score in sorted(scores.items(), key=lambda x: -x[1]):
        marker = '  ←' if choice == pred else ''
        print(f'  {choice:15s}: {score:4f}{marker}')
    print(f'Prediction: {pred}')
    print(f'Correct:    {pred == true_label}')
    print()

# === Đánh giá accuracy ===
print('-' * 40)
print('Đánh giá accuracy trên validation set...')
acc_after = evaluate_accuracy(model, tokenizer, val_preview, num_samples=100)

print()
print('=' * 60)
print('SO SANH TRUOC / SAU FINE-TUNING')
print('=' * 60)
print(f'  Before FT:  {acc_before:.2%}')
print(f'  After FT:   {acc_after:.2%}')
print(f'  Improvement: {acc_after - acc_before:+.2%}')
print()

# === Lưu model ===
SAVE_PATH = './saved_models/gpt2_nli_demo/'
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')
print('=' * 60)


---

## Tổng kết

| Metric | Before FT | After FT |
|--------|:---------:|:--------:|
| NLI Accuracy | ... | ... |
| Prompt format | `<s> Premise $ Hypothesis Relationship:\n` | same |
| Model | GPT-2 | GPT-2 |
| Fine-tuning | ✗ | ✓ |

> *Các giá trị accuracy được tính trong quá trình chạy notebook.*

### Kết luận

- **GPT-2 pretrained (zero-shot)** không được huấn luyện cho NLI → dự đoán gần như ngẫu nhiên (~33%) hoặc thiên về một class.
- **GPT-2 sau fine-tuning** học được mối quan hệ giữa premise và hypothesis thông qua prompt formatting.
- Prompt format **giống hệt nhau** trước và sau — chỉ trọng số mô hình thay đổi.
- Toàn bộ quá trình giữ nguyên kiến trúc **causal language modeling**, không thêm classification head.

### Cách dùng notebook này

1. Chạy từ đầu đến cuối để xem toàn bộ quy trình.
2. Điều chỉnh `USE_LARGER_SUBSET` để thử với 3000 mẫu (yêu cầu GPU).
3. Bật `RUN_SIMILARITY = True` để thử thêm tác vụ Semantic Similarity.
4. Model đã lưu tại `./saved_models/gpt2_nli_demo/` có thể load lại mà không cần retrain.

> **Điểm chính:** Cùng một GPT-2, cùng prompt format. Chỉ khác trọng số (fine-tuned vs. pretrained) → khác chất lượng dự đoán.

---
